Load the raw CSV (exported from Neo4j), clean it, normalize columns, extract event-level fields, assign municipality, and save events_cleaned.csv

In [1]:
# ===============================================
# Code Block 1 — Imports, Configuration, Path Checks
# ===============================================

import os
import numpy as np
import pandas as pd
import dateutil.parser

# ---- File paths ----
PATH_RAW = "../data/bpic15_eventlog_full.csv"
OUT_DIR  = "../data/processed"
OUT_FILE = os.path.join(OUT_DIR, "events_cleaned.csv")

# ---- Ensure processed directory exists ----
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Validate input file exists ----
if not os.path.exists(PATH_RAW):
    raise FileNotFoundError(f"ERROR: File not found at PATH_RAW: {PATH_RAW}")

print(f"Input file located successfully:\n{PATH_RAW}")


Input file located successfully:
../data/bpic15_eventlog_full.csv


In [2]:
# ===============================================
# Code Block 2 — Load CSV (Safe Read + Validation)
# ===============================================

try:
    # Using pandas directly (dataset is now smaller and clean)
    events = pd.read_csv(PATH_RAW, low_memory=False)
    print("CSV loaded successfully.")
except Exception as e:
    raise RuntimeError(f"Failed to load CSV: {e}")

# ---- Basic validation ----
print("\n--- Dataset Overview ---")
print("Shape:", events.shape)
print("\nColumns:", events.columns.tolist()[:25], "...")

# Show sample rows
print("\n--- First 5 Rows ---")
display(events.head())

# Ensure minimum required columns exist
required_cols = ["case_id", "timestamp", "activity"]
missing = [c for c in required_cols if c not in events.columns]

if missing:
    raise KeyError(f"ERROR: Missing required columns in CSV: {missing}")
else:
    print("\nAll required columns are present.")


CSV loaded successfully.

--- Dataset Overview ---
Shape: (262628, 10)

Columns: ['case_id', 'timestamp', 'activity', 'resource', 'cost', 'municipality', 'case_status', 'case_procedure', 'event_id', 'all_properties'] ...

--- First 5 Rows ---


,case_id,timestamp,activity,resource,cost,municipality,case_status,case_procedure,event_id,all_properties
0,3553191,2010-12-20T14:43Z,applicant is stakeholder,560604,2442.8455,BPIC15_5,G,Unknown,NaN,"{""id"":0,""labels"":[""Event""],""properties"":{""endD..."
1,3553191,2010-12-20T14:43:04Z,terminate on request,560604,2442.8455,BPIC15_5,G,Unknown,NaN,"{""id"":1,""labels"":[""Event""],""properties"":{""endD..."
2,3553191,2010-12-20T14:47:04Z,activities regular procedure,560604,2442.8455,BPIC15_5,G,Unknown,NaN,"{""id"":2,""labels"":[""Event""],""properties"":{""endD..."
3,3553191,2010-12-20T14:47:24Z,regular procedure applies,560604,2442.8455,BPIC15_5,G,Unknown,NaN,"{""id"":3,""labels"":[""Event""],""properties"":{""endD..."
4,3553191,2010-12-20T14:47:50Z,request complete,560604,2442.8455,BPIC15_5,G,Unknown,NaN,"{""id"":4,""labels"":[""Event""],""properties"":{""endD..."



All required columns are present.


In [3]:
# ===============================================
# Code Block 3 — Clean Column Names & Basic Normalization
# ===============================================

# Strip whitespace in column names
events.columns = [c.strip() for c in events.columns]

# Convert to a uniform lowercase-with-underscore format 
# (keeps compatibility but avoids spaces & case mismatch)
events.columns = [c.replace(" ", "_") for c in events.columns]

print("Normalized column names. Sample:", events.columns[:10])


Normalized column names. Sample: Index(['case_id', 'timestamp', 'activity', 'resource', 'cost', 'municipality',
       'case_status', 'case_procedure', 'event_id', 'all_properties'],
      dtype='object')


In [4]:
# ===============================================
# Code Block 4 — Robust Timestamp Parsing
# ===============================================

def parse_timestamp(val):
    """
    Parses mixed-format timestamps safely.
    Uses dateutil for ISO and non-ISO formats.
    """
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()

    # Skip non-date values
    bad_values = ["unknown", "nan", "", "null", "none", "0", "00:00.0"]
    if val.lower() in bad_values:
        return pd.NaT

    try:
        return dateutil.parser.parse(val)
    except:
        return pd.NaT

# Parse timestamp column
events["timestamp"] = events["timestamp"].apply(parse_timestamp)

print("\n--- Timestamp Parsing Validation ---")
print("Valid timestamps:", events["timestamp"].notna().sum())
print("Missing timestamps:", events["timestamp"].isna().sum())

# Show some parsed samples
print("\nParsed timestamp examples:")
display(events["timestamp"].dropna().head())



--- Timestamp Parsing Validation ---
Valid timestamps: 262628
Missing timestamps: 0

Parsed timestamp examples:


0   2010-12-20 14:43:00+00:00
1   2010-12-20 14:43:04+00:00
2   2010-12-20 14:47:04+00:00
3   2010-12-20 14:47:24+00:00
4   2010-12-20 14:47:50+00:00
Name: timestamp, dtype: datetime64[ns, tzutc()]

In [5]:
# ===============================================
# Code Block 5 — Timestamp Coverage Validation
# ===============================================

total = len(events)
valid = events["timestamp"].notna().sum()
coverage = valid / total

print(f"Timestamps present in {valid}/{total} rows ({coverage:.2%})")

if coverage < 0.90:
    print("\nWARNING: Timestamp coverage is below 90%.")
    print("This might indicate an export issue or additional timestamp columns.")
else:
    print("Timestamp coverage is sufficient for analysis.")


Timestamps present in 262628/262628 rows (100.00%)
Timestamp coverage is sufficient for analysis.


In [6]:
# ===============================================
# Code Block 6 — Remove Rows Missing Essential Fields
# ===============================================

before = len(events)

events = events[
    events["case_id"].notna() &
    events["timestamp"].notna() &
    events["activity"].notna()
].copy()

after = len(events)

print(f"Filtered unusable rows: {before - after}")
print(f"Remaining rows: {after}")


Filtered unusable rows: 0
Remaining rows: 262628


In [7]:
# ===============================================
# Code Block 7 — Standardize Key Columns
# ===============================================

events["case_id"]  = events["case_id"].astype(str)
events["activity"] = events["activity"].astype(str)

if "resource" in events.columns:
    events["resource"] = events["resource"].astype(str)
else:
    events["resource"] = "UNKNOWN"

print("Case ID / Activity / Resource standardized.")


Case ID / Activity / Resource standardized.


In [8]:
# ===============================================
# Code Block 8 — Derive Time Components
# ===============================================

events["year"]    = events["timestamp"].dt.year
events["month"]   = events["timestamp"].dt.month
events["weekday"] = events["timestamp"].dt.weekday
events["hour"]    = events["timestamp"].dt.hour

print("Derived time components added.")
display(events[["timestamp", "year", "month", "weekday", "hour"]].head())


Derived time components added.


,timestamp,year,month,weekday,hour
0,2010-12-20 14:43:00+00:00,2010,12,0,14
1,2010-12-20 14:43:04+00:00,2010,12,0,14
2,2010-12-20 14:47:04+00:00,2010,12,0,14
3,2010-12-20 14:47:24+00:00,2010,12,0,14
4,2010-12-20 14:47:50+00:00,2010,12,0,14


In [9]:
# ===============================================
# Code Block 9 — Sort Events Chronologically
# ===============================================

events = events.sort_values(["case_id", "timestamp"]).reset_index(drop=True)

print("Sorting complete. Showing sample:")
display(events.head(10))


Sorting complete. Showing sample:


,case_id,timestamp,activity,resource,cost,municipality,case_status,case_procedure,event_id,all_properties,year,month,weekday,hour
0,10002463,2014-08-05 00:00:00+00:00,register submission date request,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":136730,""labels"":[""Event""],""properties"":{...",2014,8,1,0
1,10002463,2014-08-06 00:00:00+00:00,regular procedure without MER,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137411,""labels"":[""Event""],""properties"":{...",2014,8,2,0
2,10002463,2014-08-06 00:00:00+00:00,create procedure confirmation,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137412,""labels"":[""Event""],""properties"":{...",2014,8,2,0
3,10002463,2014-08-06 00:00:00+00:00,publish,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137413,""labels"":[""Event""],""properties"":{...",2014,8,2,0
4,10002463,2014-08-06 00:00:00+00:00,create subcases completeness,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137414,""labels"":[""Event""],""properties"":{...",2014,8,2,0
5,10002463,2014-08-06 00:00:00+00:00,procedure change,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137415,""labels"":[""Event""],""properties"":{...",2014,8,2,0
6,10002463,2014-08-06 00:00:00+00:00,OLO messaging active,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137416,""labels"":[""Event""],""properties"":{...",2014,8,2,0
7,10002463,2014-08-06 00:00:00+00:00,send confirmation receipt,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137417,""labels"":[""Event""],""properties"":{...",2014,8,2,0
8,10002463,2014-08-06 00:00:00+00:00,forward to the competent authority,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137418,""labels"":[""Event""],""properties"":{...",2014,8,2,0
9,10002463,2014-08-06 00:00:00+00:00,send procedure confirmation,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137419,""labels"":[""Event""],""properties"":{...",2014,8,2,0


In [10]:
# ===============================================
# Code Block 10 — Save Cleaned Dataset
# ===============================================

events.to_csv(OUT_FILE, index=False)

print(f"Saved cleaned event log to:\n{OUT_FILE}")
print(f"Final dataset shape: {events.shape}")


Saved cleaned event log to:
../data/processed\events_cleaned.csv
Final dataset shape: (262628, 14)
